In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Olist Ecommerce - Exploratory Data Analysis\n",
    "\n",
    "This notebook performs initial exploration of the Olist datasets to understand data structure, quality, and relationships."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "from pyspark.sql import SparkSession\n",
    "from pyspark.sql.functions import col, count, when, desc\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "# Initialize Spark\n",
    "spark = SparkSession.builder \\\n",
    "    .appName(\"Olist EDA\") \\\n",
    "    .config(\"spark.sql.extensions\", \"org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions\") \\\n",
    "    .getOrCreate()\n",
    "\n",
    "print(\"Spark session created\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load and Explore Datasets"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Load customers data\n",
    "customers_df = spark.read.option(\"header\", True).csv(\"/home/hadoop-user/olist-ecommerce/datasets/olist_customers_dataset.csv\")\n",
    "print(f\"Customers: {customers_df.count()} records\")\n",
    "customers_df.printSchema()\n",
    "customers_df.show(5)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Load orders data\n",
    "orders_df = spark.read.option(\"header\", True).csv(\"/home/hadoop-user/olist-ecommerce/datasets/olist_orders_dataset.csv\")\n",
    "print(f\"Orders: {orders_df.count()} records\")\n",
    "orders_df.printSchema()\n",
    "orders_df.show(5)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Load products data\n",
    "products_df = spark.read.option(\"header\", True).csv(\"/home/hadoop-user/olist-ecommerce/datasets/olist_products_dataset.csv\")\n",
    "print(f\"Products: {products_df.count()} records\")\n",
    "products_df.printSchema()\n",
    "products_df.show(5)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Data Quality Assessment"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Check for null values\n",
    "def check_nulls(df, name):\n",
    "    null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).collect()[0]\n",
    "    print(f\"\\n{name} NULL counts:\")\n",
    "    for col_name, null_count in null_counts.asDict().items():\n",
    "        if null_count > 0:\n",
    "            print(f\"  {col_name}: {null_count}\")\n",
    "\n",
    "check_nulls(customers_df, \"Customers\")\n",
    "check_nulls(orders_df, \"Orders\")\n",
    "check_nulls(products_df, \"Products\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Check for duplicates\n",
    "def check_duplicates(df, key_col, name):\n",
    "    duplicate_count = df.groupBy(key_col).count().filter(col(\"count\") > 1).count()\n",
    "    print(f\"{name} duplicates in {key_col}: {duplicate_count}\")\n",
    "\n",
    "check_duplicates(customers_df, \"customer_id\", \"Customers\")\n",
    "check_duplicates(orders_df, \"order_id\", \"Orders\")\n",
    "check_duplicates(products_df, \"product_id\", \"Products\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Business Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Customer distribution by state\n",
    "customers_df.groupBy(\"customer_state\").count().orderBy(desc(\"count\")).show(10)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Order status distribution\n",
    "orders_df.groupBy(\"order_status\").count().orderBy(desc(\"count\")).show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Key Insights\n",
    "- Most customers are from SP, RJ, MG states\n",
    "- Order status shows delivered orders are majority\n",
    "- Some datasets have null values that need handling"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}